## import module

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import ConcatDataset
from torch.utils.data import random_split
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Model design

In [ ]:
class PCAProjector(nn.Module):
    def __init__(self, input_dim, output_dim=3):
        super(PCAProjector, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)

class MultiHeadConv(nn.Module):
    def __init__(self, input_dim, output_dim, num_heads=4):
        super(MultiHeadConv, self).__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=input_dim, out_channels=output_dim, kernel_size=ks, padding=ks//2)
            for ks in [3, 5, 7][:num_heads]
        ])
        self.output_dim = output_dim * len(self.convs)

    def forward(self, x):
        x = x.permute(0, 2, 1)  # (batch, input_dim, seq_len)
        out = [conv(x) for conv in self.convs] 
        out = torch.cat(out, dim=1)  # (batch, out_dim * num_heads, seq_len)
        out = out.permute(0, 2, 1)  # (batch, seq_len, out_dim * num_heads)
        return out

class AttentionReduce(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(AttentionReduce, self).__init__()
        self.fc = nn.Linear(input_dim, output_dim)
        self.attn = nn.Linear(input_dim, 1)

    def forward(self, x):
        attn_weights = torch.softmax(self.attn(x), dim=1)  # (batch, seq_len, 1)
        x_weighted = (x * attn_weights).sum(dim=1)  # (batch, input_dim)
        return self.fc(x_weighted)  # (batch, output_dim)

class FeatureExtractor(nn.Module):
    def __init__(self, classes=3, feat_dim=52, pca_out=3, lstm_hidden=64, conv_out=32, num_heads=3, final_dim=128):
        super(FeatureExtractor, self).__init__()
        self.pca = PCAProjector(feat_dim, pca_out)
        self.lstm = nn.LSTM(pca_out, lstm_hidden, batch_first=True)
        self.time_reduce = nn.Linear(lstm_hidden, final_dim)
        self.drop = nn.Dropout(0.6)

        self.multi_conv = MultiHeadConv(feat_dim, conv_out, num_heads)
        self.attn_reduce = AttentionReduce(conv_out * num_heads, final_dim)

        self.classifier = nn.Linear(256, classes)
        
        self.decoder = nn.Sequential(
            nn.Linear(1024 * 2, 2048),
            nn.ReLU(),
            nn.Linear(2048, 200 * 104),
            nn.Tanh()
        )

    def extract_time(self, x):
        x = self.pca(x)
        out, _ = self.lstm(x)
        return self.time_reduce(out[:, -1, :]) 

    def extract_space(self, x):
        x = self.multi_conv(x)
        return self.attn_reduce(x)
    
    def forward(self, x):
        amp = x[:, :, :52]  # amplitude (batch, 200, 52)
        pha = x[:, :, 52:]  # phase (batch, 200, 52)

        f1 = self.extract_space(amp)
        f2 = self.extract_time(pha)

        f = torch.cat([f1, f2], dim=1)  # (batch, final_dim * 2)
        f = self.drop(f)
        features = f.to(device)
        
        identity_logits = self.classifier(features)
        return identity_logits

## Load data

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, data, labels):
        """
        Args:
            data: numpy array (num_samples, num_features)
            labels: numpy array (num_samples,)
        """
        self.data = torch.FloatTensor(data) 
        self.labels = torch.LongTensor(labels) if labels.dtype == int else torch.FloatTensor(labels)
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

loaded1 = np.load('detection/dataset_drone1.npz')
loaded2 = np.load('detection/dataset_camera1.npz')
loaded3 = np.load('detection/dataset_router1.npz')

loaded4 = np.load('detection/dataset_camera2.npz')
loaded5 = np.load('detection/dataset_router2.npz')
loaded6 = np.load('detection/dataset_drone2.npz')

loaded7 = np.load('detection/dataset_drone3.npz')
loaded8 = np.load('detection/dataset_drone4.npz')
loaded9 = np.load('detection/dataset_drone5.npz')

# 提取数据和标签

data1, labels1 = loaded1['data'], loaded1['labels']
data2, labels2 = loaded2['data'], loaded2['labels']
data3, labels3 = loaded3['data'], loaded3['labels']

data4, labels4 = loaded4['data'], loaded4['labels']
data5, labels5 = loaded5['data'], loaded5['labels']
data6, labels6 = loaded6['data'], loaded6['labels']

data7, labels7 = loaded7['data'], loaded7['labels']
data8, labels8 = loaded8['data'], loaded8['labels']
data9, labels9 = loaded9['data'], loaded9['labels']

batch_size = 32
data_train = np.concatenate([data1[:300,:,:], data2[:300,:,:], data3[:300,:,:],], axis=0)
labels_train = np.concatenate([labels1[:300,:], labels2[:300,:], labels3[:300,:]], axis=0)
dataset0 = CustomDataset(data_train,labels_train)
train_loader = DataLoader(dataset0, batch_size=batch_size, shuffle=True, num_workers=0)  

test1_data = data1[300:,:,:]
test1_label = labels1[300:,:]
test2_data = data2[300:,:,:]
test2_label = labels2[300:,:]
test3_data = data3[300:,:,:]
test3_label = labels3[300:,:]
dataset1 = CustomDataset(test1_data,test1_label)
dataset2= CustomDataset(test2_data,test2_label)
dataset3= CustomDataset(test3_data,test3_label)# 
test_loader1 = DataLoader(dataset1, batch_size=batch_size, shuffle=False)
test_loader2 = DataLoader(dataset2, batch_size=batch_size, shuffle=False)
test_loader3 = DataLoader(dataset3, batch_size=batch_size, shuffle=False)
dataset4 = CustomDataset(data4,labels4)
dataset5= CustomDataset(data5,labels5)
dataset6= CustomDataset(data6,labels6)
test_loader4 = DataLoader(dataset4, batch_size=batch_size, shuffle=False)
test_loader5 = DataLoader(dataset5, batch_size=batch_size, shuffle=False)
test_loader6 = DataLoader(dataset6, batch_size=batch_size, shuffle=False)

dataset7= CustomDataset(data7,labels7)
dataset8= CustomDataset(data8,labels8)
dataset9= CustomDataset(data9,labels9)
test_loader7 = DataLoader(dataset7, batch_size=batch_size, shuffle=False)
test_loader8 = DataLoader(dataset8, batch_size=batch_size, shuffle=False)
test_loader9 = DataLoader(dataset9, batch_size=batch_size, shuffle=False)

print("data has loaded")

## Train and test

In [ ]:
# ==== 训练与测试 ====
def train_detection(model, train_loader, num_epochs=10, lr=0.001):
    criterion_id = nn.CrossEntropyLoss()
    criterion_recon = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    all_labels = torch.tensor([15,20,30])
    unique_labels = torch.unique(all_labels)
    label_mapping = {label.item(): idx for idx, label in enumerate(unique_labels)}
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for data, labels in progress_bar:
            labels = torch.tensor([label_mapping[label.item()] for label in labels],device=labels.device)
            data  = data.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            
            # identity_logits = model(data)
            # loss = criterion_id(identity_logits, labels)
            
            identity_logits, dispelled_features, reconstructed = model(data)
            loss_id = criterion_id(identity_logits, labels)
            loss_recon = criterion_recon(reconstructed, data)
            loss_adv = -torch.mean(torch.log(dispelled_features + 1e-6))
            loss = loss_id + loss_recon + 0.1 * loss_adv
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            progress_bar.set_postfix(loss=running_loss / len(train_loader))
    
    print("Training Complete")

def test_detection(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    all_labels = torch.tensor([15, 20, 30])
    unique_labels = torch.unique(all_labels)
    label_mapping = {label.item(): idx for idx, label in enumerate(unique_labels)}
    with torch.no_grad():
        for data, labels in test_loader:
            labels = torch.tensor([label_mapping[label.item()] for label in labels],device=labels.device)
            data, labels = data.to(device), labels.to(device)
            identity_logits, _, _ = model(data)
            predictions = torch.argmax(identity_logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    print(f"Test Accuracy: {100 * correct / total:.2f}%")

## Evaluation

In [ ]:
model = FeatureExtractor().to(device)

train_detection(model, train_loader, num_epochs=40)
# model.load_state_dict(torch.load('model/model_clf_f3_unknow.pth'))

test_detection(model, test_loader1)
test_detection(model, test_loader2)
test_detection(model, test_loader3)

test_detection(model, test_loader4)
test_detection(model, test_loader5)
test_detection(model, test_loader6)
test_detection(model, test_loader7)
test_detection(model, test_loader8)
test_detection(model, test_loader9)

## Dr. Defender

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.mixture import GaussianMixture
from scipy.signal import stft, convolve
from scipy.stats import norm

loaded55 = np.load('dataset_drone1.npz')
loaded66 = np.load('dataset_camera1.npz')
loaded77 = np.load('dataset_router1.npz')
data55, labels55 = loaded55['data'], loaded55['labels']
data66, labels66 = loaded66['data'], loaded66['labels']
data77, labels77 = loaded77['data'], loaded77['labels']
test1_amp= data55[300:,:,:52]
test1_pha= data55[300:,:,52:]
test1_label = labels55[300:,:]
test1_mean = np.mean(test1_data,axis=2)

test2_amp = data66[300:,:,:52]
test2_pha = data66[300:,:,52:]
test2_mean = np.mean(test2_data,axis=2)

test3_amp = data77[300:,:,:52]
test3_pha = data77[300:,:,52:]
test3_label = labels77[300:,:]
test3_mean = np.mean(test3_data,axis=2)

fs = 200 
gmm_threshold = 0.5
window_size = 5  # seconds

def ps_anomaly_detection(samples, threshold=gmm_threshold):
    # samples: (200, 2) => [amplitude_mean, phase_mean]
    gmm = GaussianMixture(n_components=2, covariance_type='full')
    gmm.fit(samples)
    prob = gmm.score_samples(samples)
    ai = -np.log(prob)
    anomalies = ai > threshold
    return anomalies

def tfd_anomaly_detection(signal, low_f=0, high_f=30, k=3):
    f, t, Zxx = stft(signal, fs=fs, nperseg=64)
    power = np.abs(Zxx)**2
    idx = (f >= low_f) & (f <= high_f)
    energy = np.sum(power[idx, :], axis=0)
    mean_energy = np.mean(energy)
    std_energy = np.std(energy)
    anomalies = energy > (mean_energy + k * std_energy)
    return anomalies.any()

def template_matching(P1, P2):
    t = np.linspace(-window_size/2, window_size/2, len(P1))
    T1 = norm.pdf(t, 0, 1)
    T2 = norm.pdf(t, -window_size/4, 1) + norm.pdf(t, window_size/4, 1)
    match = lambda X, Y: convolve(X, T1, mode='same') * convolve(Y, T2, mode='same')
    MR = np.maximum(match(P1, P2), match(P2, P1))
    return (MR > 1.3).any()

def fixed_freq_detector(spectrogram, target_freq=35):
    freqs = np.linspace(0, fs/2, spectrogram.shape[0])
    idx = np.argmin(np.abs(freqs - target_freq))
    line_energy = spectrogram[idx, :]
    return np.mean(line_energy) > np.percentile(spectrogram, 95)

def detect_drone(amp, phs):
    # amp, phs: (200, 52)
    amp_mean = np.mean(amp, axis=1)  # (200,)
    phs_mean = np.mean(phs, axis=1)
    features = np.stack([amp_mean, phs_mean], axis=-1)  # (200, 2)

    # Step 1: Anomaly Detection
    anomaly_ps = ps_anomaly_detection(features)
    anomaly_tfd = tfd_anomaly_detection(amp_mean)

    if not (anomaly_ps.any() or anomaly_tfd):
        return False  # no anomaly, no drone

    # Step 2: Drone Discrimination
    f, t, Zxx = stft(amp_mean, fs=fs, nperseg=256)
    power = np.abs(Zxx)**2
    P1 = np.sum(power[(f >= 2) & (f <= 30)], axis=0)
    P2 = np.sum(power[(f >= 0) & (f < 2)], axis=0)
    smf_result = template_matching(P1, P2)
    sfe_result = fixed_freq_detector(power)
    return smf_result or sfe_result


warnings.filterwarnings('ignore')
num_samples = data55.shape[0]-300
drone_count = 0
for i in range(num_samples):
    amp_i = test1_amp[i]  # (200, 52)
    phs_i = test1_pha[i]      # (200, 52)
    if detect_drone(amp_i, phs_i):
        drone_count += 1
drone_ratio = drone_count / num_samples * 100
# print(f"🚨 Detected drones in {drone_count} out of {num_samples} samples.")
print(f"📊 Drone detection ratio: {drone_ratio:.2f}%")
num_samples = data66.shape[0]-300
drone_count = 0
for i in range(num_samples):
    amp_i = test1_amp[i]  # (200, 52)
    phs_i = test1_pha[i]      # (200, 52)
    if detect_drone(amp_i, phs_i):
        drone_count += 1
drone_ratio = drone_count / num_samples * 100
# print(f"🚨 Detected drones in {drone_count} out of {num_samples} samples.")
print(f"📊 Drone detection ratio: {drone_ratio:.2f}%")
num_samples = data77.shape[0]-300
drone_count = 0
for i in range(num_samples):
    amp_i = test1_amp[i]  # (200, 52)
    phs_i = test1_pha[i]      # (200, 52)
    if detect_drone(amp_i, phs_i):
        drone_count += 1
drone_ratio = drone_count / num_samples * 100
# print(f"🚨 Detected drones in {drone_count} out of {num_samples} samples.")
print(f"📊 Drone detection ratio: {drone_ratio:.2f}%")

## ML model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import get_window
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

loaded55 = np.load('dataset_drone1.npz')
loaded66 = np.load('dataset_camera1.npz')
loaded77 = np.load('dataset_router1.npz')
data55, labels55 = loaded55['data'], loaded55['labels']
data66, labels66 = loaded66['data'], loaded66['labels']
data77, labels77 = loaded77['data'], loaded77['labels']

csi_data = np.concatenate([data55[:300,:], data66[:300,:], data77[:300,:]], axis=0)
labels = np.concatenate([labels55[:300,:], labels66[:300,:], labels77[:300,:]], axis=0)
amplitude_data  = csi_data[:, :, :52]  # Shape: (sample_num, 200, 52)
avg_amplitude_signals = np.mean(amplitude_data, axis=2) # Shape: (sample_num, 200)

test1_data = data55[300:,:,:52]
test1_label = labels55[300:,:]
test1_mean = np.mean(test1_data,axis=2)

test2_data = data66[300:,:,:52]
test2_label = labels66[300:,:]
test2_mean = np.mean(test2_data,axis=2)

test3_data = data77[300:,:,:52]
test3_label = labels77[300:,:]
test3_mean = np.mean(test3_data,axis=2)

N = avg_amplitude_signals.shape[1]  # 采样点数，即200
Fs = 200.0  
T = 1.0 / Fs  

window = get_window('hann', N)

freq_bands = [
    (1, 10),    # 1-10 Hz
    (10, 20),   # 10-20 Hz
    (20, 30),   # 20-30 Hz
    (30, 40),   # 30-40 Hz
    (40, 50),   # 40-50 Hz
    (50, 75),   # 50-75 Hz
    (75, 100)   # 75-100 Hz
]
num_bands = len(freq_bands)
all_features = []
xf = fftfreq(N, T)[:N//2] 

for i in range(avg_amplitude_signals.shape[0]):
    sample_signal = avg_amplitude_signals[i, :]
    sample_signal_windowed = sample_signal * window

    yf = fft(sample_signal_windowed)
    psd = (2.0/N * np.abs(yf[0:N//2]))**2 # Power Spectral Density

    band_powers = []
    for band_low, band_high in freq_bands:
        idx_band = np.where((xf >= band_low) & (xf < band_high))[0]
        if len(idx_band) > 0:
            avg_power_in_band = np.mean(psd[idx_band])
        else:
            avg_power_in_band = 0 
        band_powers.append(avg_power_in_band)
    all_features.append(band_powers)

X = np.array(all_features) # Shape: (sample_num, num_bands)
y = labels

all_features = []
for i in range(test1_mean.shape[0]):
    sample_signal = test1_mean[i, :]
    sample_signal_windowed = sample_signal * window 
    yf = fft(sample_signal_windowed)
    psd = (2.0/N * np.abs(yf[0:N//2]))**2 # Power Spectral Density
    band_powers = []
    for band_low, band_high in freq_bands:
        idx_band = np.where((xf >= band_low) & (xf < band_high))[0]
        if len(idx_band) > 0:
            avg_power_in_band = np.mean(psd[idx_band])
        else:
            avg_power_in_band = 0 
        band_powers.append(avg_power_in_band)
    all_features.append(band_powers)
X_test1 = np.array(all_features) # Shape: (sample_num, num_bands)
y_test1 = test1_label

all_features = []
for i in range(test2_mean.shape[0]):
    sample_signal = test2_mean[i, :]
    sample_signal_windowed = sample_signal * window 
    yf = fft(sample_signal_windowed)
    psd = (2.0/N * np.abs(yf[0:N//2]))**2 # Power Spectral Density
    band_powers = []
    for band_low, band_high in freq_bands:
        idx_band = np.where((xf >= band_low) & (xf < band_high))[0]
        if len(idx_band) > 0:
            avg_power_in_band = np.mean(psd[idx_band])
        else:
            avg_power_in_band = 0 
        band_powers.append(avg_power_in_band)
    all_features.append(band_powers)
X_test2 = np.array(all_features) # Shape: (sample_num, num_bands)
y_test2 = test2_label

all_features = []
for i in range(test3_mean.shape[0]):
    sample_signal = test3_mean[i, :]
    sample_signal_windowed = sample_signal * window
    yf = fft(sample_signal_windowed)
    psd = (2.0/N * np.abs(yf[0:N//2]))**2 # Power Spectral Density
    band_powers = []
    for band_low, band_high in freq_bands:
        idx_band = np.where((xf >= band_low) & (xf < band_high))[0]
        if len(idx_band) > 0:
            avg_power_in_band = np.mean(psd[idx_band])
        else:
            avg_power_in_band = 0 
        band_powers.append(avg_power_in_band)
    all_features.append(band_powers)
X_test3 = np.array(all_features) # Shape: (sample_num, num_bands)
y_test3 = test3_label

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X)
X_test_scaled1 = scaler.transform(X_test1)
X_test_scaled2 = scaler.transform(X_test2)
X_test_scaled3 = scaler.transform(X_test3)

classifiers = {
    "SVM (RBF Kernel)": SVC(kernel='rbf', probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}
for name, clf in classifiers.items():
    print(f"\n--- Training and Evaluating {name} ---")
    clf.fit(X_train_scaled, y)
    
    y_pred1 = clf.predict(X_test_scaled1)
    accuracy1 = accuracy_score(y_test1, y_pred1)
    print(f"Accuracy1: {accuracy1:.4f}")
    y_pred2 = clf.predict(X_test_scaled2)
    accuracy2 = accuracy_score(y_test2, y_pred2)
    print(f"Accuracy2: {accuracy2:.4f}")
    y_pred3 = clf.predict(X_test_scaled3)
    accuracy3 = accuracy_score(y_test3, y_pred3)
    print(f"Accuracy3: {accuracy3:.4f}")

def plot_sample_spectrum_with_bands(sample_idx_to_plot, X_sample_features, xf_axis, psd_sample, bands_def):
    plt.figure(figsize=(10, 6))
    plt.plot(xf_axis, psd_sample, label=f'PSD Sample {sample_idx_to_plot}')
    plt.title(f'Power Spectrum for Sample {sample_idx_to_plot} (Label: {y[sample_idx_to_plot]})')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Power Spectral Density')

    colors = plt.cm.get_cmap('viridis', len(bands_def))
    for i, (band_low, band_high) in enumerate(bands_def):
        idx_band = np.where((xf_axis >= band_low) & (xf_axis < band_high))[0]
        if len(idx_band) > 0:
            band_power_val = X_sample_features[i]
            plt.fill_between(xf_axis[idx_band], 0, psd_sample[idx_band], alpha=0.3, color=colors(i),
                             label=f'Band {band_low}-{band_high} Hz, Power: {band_power_val:.2e}')
    plt.legend(fontsize='small', loc='upper right')
    plt.grid(True)
    plt.xlim(0, Fs/2)
    plt.show()
